## The Parity Oracle Attack

RSA is **multiplicatively homomorphic**:

    Enc(m · s) = Enc(m) · Enc(s) = c · s^e  (mod n)

An attacker who can ask a server "is the decryption of this ciphertext even or odd?"
can recover any plaintext with only **ceil(log₂ n) queries** — one bit per query.

### How it works

At step k the attacker submits c' = c · (2^e)^k mod n. The server decrypts and
returns the LSB of m·2^k mod n. Because n is odd, this bit equals the parity of
floor(m·2^k / n) — effectively one bit of the binary expansion of m/n.
After ~log₂(n) queries, m is fully pinpointed.

### This implementation

A `Ranges` object tracks the set of candidate plaintexts as disjoint integer
intervals. Each oracle response halves the candidate set by filtering out
inconsistent values of q = floor(m·2^k / n).

**Reference:** Bleichenbacher, D. (1998). *Chosen Ciphertext Attacks Against
Protocols Based on the RSA Encryption Standard PKCS #1.* CRYPTO 1998, pp. 1–12.

In [ ]:
import math
from Crypto.Util.number import getPrime

E = 65537

while True:
    p = getPrime(128)
    q = getPrime(128)
    if p != q and math.gcd(E, (p - 1) * (q - 1)) == 1:
        break

n   = p * q
phi = (p - 1) * (q - 1)
d   = pow(E, -1, phi)

print(f"Key size  : {n.bit_length()} bits")
print(f"n = {hex(n)[:26]}...")
print(f"p = {hex(p)[:26]}...")
print(f"q = {hex(q)[:26]}...")
print(f"e = {E}")

In [ ]:
from rsa_attacks.parity_oracle import SimulatedOracle

PLAINTEXT = b"RSA oracle"
m = int.from_bytes(PLAINTEXT, "big")
c = pow(m, E, n)

oracle = SimulatedOracle(n, E, d)

print(f"Plaintext : {PLAINTEXT!r}  ->  integer {m}")
print(f"Encrypted : {hex(c)[:26]}...")
print()
print(f"oracle(c)   = {oracle(c)}   (LSB of plaintext m = {m % 2})")
print(f"oracle(2c)  = {oracle(pow(2, E, n) * c % n)}   (LSB of 2m mod n)")

In [ ]:
import time
from rsa_attacks.parity_oracle import Ranges, _narrow_interval

call_count = 0
log_rows   = []

interval   = Ranges((0, n - 1))
multiplier = 1
result     = None
elapsed    = 0.0

start = time.perf_counter()

for step in range(n.bit_length() + 5):
    c_prime = (c * pow(multiplier, E, n)) % n
    bit = oracle(c_prime)
    call_count += 1

    if multiplier > 1:
        interval = _narrow_interval(interval, multiplier, n, bit)

    # Candidate set width expressed as a bit count
    total_width = sum(hi - lo + 1 for lo, hi in interval._intervals)
    bits_left   = total_width.bit_length()
    log_rows.append((step, bit, len(interval._intervals), bits_left))

    result = interval.to_int()
    if result is not None:
        elapsed = time.perf_counter() - start
        break

    multiplier *= 2

# Print every 10th row plus the final row
print(f"{'Step':>4}  {'bit':>3}  {'intervals':>9}  {'bits left':>9}")
print("-" * 35)
rows_to_show = log_rows[::10]
if log_rows and log_rows[-1] not in rows_to_show:
    rows_to_show = rows_to_show + [log_rows[-1]]
for row in rows_to_show:
    print(f"{row[0]:>4}  {row[1]:>3}  {row[2]:>9}  {row[3]:>9}")

print()
print(f"Oracle calls : {call_count}")
print(f"Time         : {elapsed:.4f}s")
print()
print(f"Recovered    : {result}")
print(f"Expected     : {m}")
print(f"Match        : {result == m}")
recovered_bytes = result.to_bytes((result.bit_length() + 7) // 8, "big")
print(f"Plaintext    : {recovered_bytes}")

## The Adaptive Chosen-Ciphertext Model

This attack is a **CCA2 (adaptive chosen-ciphertext attack)**:

- **Chosen ciphertext**: the attacker crafts arbitrary ciphertexts.
- **Adaptive**: each query may depend on all previous oracle responses.
- **Oracle model**: the server reveals only one bit per decryption.

### Why padding prevents this attack

PKCS#1 v1.5 and OAEP add random bytes before encrypting. When the attacker submits
a blinded ciphertext c' = c · s^e mod n, the server decrypts to m·s mod n, which
almost never has valid padding. The server's *padding error* response is what leaks
information in Bleichenbacher's original 1998 attack — not the plaintext parity.

The parity oracle shown here requires the server to reveal `plaintext % 2` directly.
No modern TLS implementation does this; the lesson is that **any single bit** of
information about the decrypted plaintext is sufficient to mount a full recovery.

### Complexity

- Oracle queries: exactly ceil(log₂ n) — one per bit of the modulus.
- For a 2048-bit key: 2048 queries; each requires one server-side modular exponentiation.
- Total attack cost: O(log n) × cost-of-one-decryption.

This attack is impractical against padded RSA in real systems, but it demonstrates
precisely why the CCA2 model matters and why OAEP padding is required by modern
standards (PKCS#1 v2.2, RFC 8017).